# Cálculo multiescala del nSPEI e identificación de eventos de sequía

## Código para la reproducción de tablas y figuras de la tesis

El cuaderno contiene la cadena computacional necesaria para calcular el nSPEI en escalas de 3, 6 y 12 meses, identificar eventos de sequía y reproducir las tablas y figuras empleadas en la investigación.

Se conservaron únicamente los procedimientos necesarios para los resultados presentados en la tesis. Las rutas fueron adecuadas a la estructura reproducible del proyecto en Windows.


## 1. Paquetes


In [ ]:
if (!require("pacman")) install.packages("pacman")  # Instala pacman si no está disponible.

pacman::p_load(
  tidyverse,   # Manipulación y visualización de datos.
  lubridate,   # Manejo de fechas.
  janitor,     # Limpieza de nombres de variables.
  zoo,         # Cálculo de acumulados móviles.
  scales,      # Herramientas auxiliares para escalas gráficas.
  grid         # Elementos gráficos auxiliares.
)


## 2. Rutas del proyecto


In [ ]:
# ============================================================
# RUTAS DEL PROYECTO - BLOQUE 02
# ============================================================

encontrar_raiz_proyecto <- function(inicio = getwd()) {
  marcadores <- c("01_datos", "02_scripts", "03_resultados", "04_documentacion")
  es_raiz <- function(x) all(dir.exists(file.path(x, marcadores)))

  raiz_env <- Sys.getenv("TESIS_SEQUIAS_ROOT", unset = "")
  if (nzchar(raiz_env)) {
    raiz_env <- normalizePath(raiz_env, winslash = "/", mustWork = FALSE)
    if (es_raiz(raiz_env)) return(raiz_env)
  }

  ruta <- normalizePath(inicio, winslash = "/", mustWork = TRUE)

  repeat {
    if (es_raiz(ruta)) return(ruta)

    padre <- dirname(ruta)
    if (identical(padre, ruta)) {
      stop(
        "No se encontró la raíz del proyecto. ",
        "Ejecute Jupyter dentro del repositorio o defina TESIS_SEQUIAS_ROOT."
      )
    }

    ruta <- padre
  }
}

ruta_base <- encontrar_raiz_proyecto()

ruta_raw        <- file.path(ruta_base, "01_datos", "raw")          # Datos originales.
ruta_inter      <- file.path(ruta_base, "01_datos", "intermedios")  # Datos intermedios.
ruta_procesados <- file.path(ruta_base, "01_datos", "procesados")   # Datos procesados.

ruta_tablas <- file.path(
  ruta_base, "03_resultados", "02_nspei_eventos", "tablas"
)  # Tablas generadas en este bloque.

ruta_figuras <- file.path(
  ruta_base, "03_resultados", "02_nspei_eventos", "figuras"
)  # Figuras generadas en este bloque.

archivo_base <- file.path(
  ruta_raw,
  "Base_Datos_Final_ETo_Pivot.csv"
)  # Base hidroclimática original.

archivo_cluster <- file.path(
  ruta_base,
  "03_resultados",
  "01_regionalizacion",
  "tablas",
  "estaciones_clusters_ORIGINAL.csv"
)  # Clasificación de estaciones utilizada en la tesis.

dir.create(ruta_inter, recursive = TRUE, showWarnings = FALSE)        # Crea carpeta de datos intermedios.
dir.create(ruta_procesados, recursive = TRUE, showWarnings = FALSE)   # Crea carpeta de datos procesados.
dir.create(ruta_tablas, recursive = TRUE, showWarnings = FALSE)       # Crea carpeta de tablas.
dir.create(ruta_figuras, recursive = TRUE, showWarnings = FALSE)      # Crea carpeta de figuras.

cat("Base hidroclimática:", file.exists(archivo_base), "\n")          # Verifica la base de entrada.
cat("Clusters:", file.exists(archivo_cluster), "\n")                  # Verifica la clasificación regional.




## 3. Lectura e integración de datos


In [ ]:
df <- read.csv(archivo_base, stringsAsFactors = FALSE) %>%  # Importa la base hidroclimática.
  clean_names() %>%                                         # Estandariza nombres de variables.
  mutate(
    fecha = as.Date(fecha),                                 # Convierte la fecha al formato de R.
    anio  = year(fecha),                                    # Extrae el año.
    mes   = month(fecha)                                    # Extrae el mes.
  )

clusters <- read.csv(archivo_cluster, stringsAsFactors = FALSE) %>%  # Importa los clusters.
  clean_names() %>%
  select(estacion, cluster_ward)                            # Conserva estación y cluster Ward.

df <- df %>%
  left_join(clusters, by = "estacion")                     # Integra el cluster de cada estación.


## 4. Balance hídrico y acumulados móviles


In [ ]:
df <- df %>%
  mutate(
    balance_hidrico = pt101 - e_to_mensual                 # Precipitación menos evapotranspiración.
  )

df_acum <- df %>%
  arrange(estacion, fecha) %>%                              # Ordena cronológicamente cada estación.
  group_by(estacion) %>%
  mutate(
    bh_3  = rollapply(balance_hidrico, width = 3,  FUN = sum, align = "right", fill = NA, na.rm = FALSE),   # Acumulado 3 meses.
    bh_6  = rollapply(balance_hidrico, width = 6,  FUN = sum, align = "right", fill = NA, na.rm = FALSE),   # Acumulado 6 meses.
    bh_12 = rollapply(balance_hidrico, width = 12, FUN = sum, align = "right", fill = NA, na.rm = FALSE)    # Acumulado 12 meses.
  ) %>%
  ungroup()


## 5. Cálculo no paramétrico del nSPEI


In [ ]:
calc_nspei_np <- function(x) {                              # Estandariza empíricamente cada serie.
  ok <- is.finite(x)
  out <- rep(NA_real_, length(x))

  if (sum(ok) < 10) return(out)                             # Exige un mínimo de datos válidos.

  ranks <- rank(x[ok], ties.method = "average")             # Obtiene rangos empíricos.
  n <- sum(ok)

  p <- (ranks - 0.44) / (n + 0.12)                         # Calcula probabilidades de posición.
  p <- pmin(pmax(p, 1e-6), 1 - 1e-6)                       # Evita probabilidades 0 y 1.

  out[ok] <- qnorm(p)                                       # Transforma a escala normal estándar.
  out
}

df_nspei <- df_acum %>%
  group_by(estacion, mes) %>%                               # Estandariza por estación y mes calendario.
  mutate(
    nspei_3  = calc_nspei_np(bh_3),                         # nSPEI a 3 meses.
    nspei_6  = calc_nspei_np(bh_6),                         # nSPEI a 6 meses.
    nspei_12 = calc_nspei_np(bh_12)                         # nSPEI a 12 meses.
  ) %>%
  ungroup()

write.csv(
  df_nspei,
  file.path(ruta_inter, "base_nspei_estaciones.csv"),
  row.names = FALSE
)


## 6. Comparación multiescala del nSPEI


In [ ]:
# Función para crear la figura multiescala por estación
graficar_nspei_multiescala <- function(df_nspei, estacion_obj) {

  df_plot <- df_nspei %>%
    filter(estacion == estacion_obj) %>%
    arrange(fecha) %>%
    select(fecha, estacion, nspei_3, nspei_6, nspei_12) %>%
    pivot_longer(
      cols = c(nspei_3, nspei_6, nspei_12),
      names_to = "escala",
      values_to = "nspei"
    ) %>%
    mutate(
      escala = factor(
        escala,
        levels = c("nspei_3", "nspei_6", "nspei_12"),
        labels = c("nSPEI-3", "nSPEI-6", "nSPEI-12")
      ),
      humedo = if_else(nspei > 0, nspei, 0),
      seco   = if_else(nspei < 0, nspei, 0)
    )

  umbrales <- tibble(
    nivel = factor(
      c("D1: Moderada (-0.8)",
        "D2: Severa (-1.3)",
        "D3: Extrema (-1.6)",
        "D4: Excepcional (-2.0)"),
      levels = c(
        "D1: Moderada (-0.8)",
        "D2: Severa (-1.3)",
        "D3: Extrema (-1.6)",
        "D4: Excepcional (-2.0)"
      )
    ),
    y = c(-0.8, -1.3, -1.6, -2.0)
  )

  p <- ggplot(df_plot, aes(x = fecha)) +
    geom_area(aes(y = humedo, fill = "Superávit (Húmedo)"),
              alpha = 0.95, na.rm = TRUE) +
    geom_area(aes(y = seco, fill = "Déficit (Sequía)"),
              alpha = 0.95, na.rm = TRUE) +
    geom_line(aes(y = nspei), color = "grey30", linewidth = 0.22, alpha = 0.75, na.rm = TRUE) +

    geom_hline(yintercept = 0, color = "grey35", linewidth = 0.9) +

    geom_hline(
      data = umbrales,
      aes(yintercept = y, linetype = nivel, color = nivel),
      linewidth = 0.95,
      show.legend = TRUE
    ) +

    facet_wrap(~ escala, ncol = 1, scales = "fixed") +

    scale_fill_manual(
      name = NULL,
      values = c(
        "Déficit (Sequía)"   = "#F05A5A",
        "Superávit (Húmedo)" = "#5DA5DA"
      ),
      breaks = c("Déficit (Sequía)", "Superávit (Húmedo)")
    ) +

    scale_linetype_manual(
      name = NULL,
      values = c(
        "D1: Moderada (-0.8)"    = "dashed",
        "D2: Severa (-1.3)"      = "dashed",
        "D3: Extrema (-1.6)"     = "dotted",
        "D4: Excepcional (-2.0)" = "dotted"
      )
    ) +

    scale_color_manual(
      name = NULL,
      values = c(
        "D1: Moderada (-0.8)"    = "#F4A300",
        "D2: Severa (-1.3)"      = "#E66100",
        "D3: Extrema (-1.6)"     = "#C51B7D",
        "D4: Excepcional (-2.0)" = "#B2182B"
      )
    ) +

    scale_x_date(
      limits = as.Date(c("1970-01-01", "2023-12-01")),
      breaks = seq(as.Date("1970-01-01"), as.Date("2023-01-01"), by = "4 years"),
      date_labels = "%Y",
      expand = c(0, 0)
    ) +

    scale_y_continuous(
      limits = c(-2.8, 2.8),
      breaks = seq(-2.5, 2.5, by = 0.5),
      expand = expansion(mult = c(0.01, 0.01))
    ) +

    labs(
      title = paste0("Comparación multiescala del nSPEI - Estación: ", estacion_obj),
      subtitle = "Serie temporal mensual de nSPEI-3, nSPEI-6 y nSPEI-12",
      x = NULL,
      y = "Índice estandarizado"
    ) +

    guides(
      fill = guide_legend(
        order = 1,
        nrow = 1,
        byrow = TRUE,
        override.aes = list(alpha = 1)
      ),
      color = guide_legend(
        order = 2,
        nrow = 1,
        byrow = TRUE
      ),
      linetype = "none"
    ) +

    theme_minimal(base_size = 12) +
    theme(
      plot.title = element_text(face = "bold", hjust = 0.5, size = 16),
      plot.subtitle = element_text(hjust = 0.5, size = 11, color = "grey30"),
      axis.text.x = element_text(angle = 45, hjust = 1, size = 9.5),
      axis.text.y = element_text(size = 9.5),
      axis.title.y = element_text(face = "bold", size = 12),
      strip.text = element_text(face = "bold", size = 12),
      strip.background = element_rect(fill = "grey92", color = "grey70"),
      panel.grid.minor = element_blank(),
      panel.grid.major.x = element_line(color = "grey86", linewidth = 0.35),
      panel.grid.major.y = element_line(color = "grey84", linewidth = 0.35),
      legend.position = "bottom",
      legend.direction = "horizontal",
      legend.box = "vertical",
      legend.background = element_blank(),
      legend.margin = margin(t = 4, b = 2),
      legend.box.margin = margin(t = 4),
      legend.key.width = unit(2.0, "lines"),
      legend.key.height = unit(0.9, "lines"),
      legend.text = element_text(size = 10),
      plot.margin = margin(10, 12, 10, 10)
    )

  return(p)
}


In [ ]:
# Figuras multiescala utilizadas en la tesis.
for (est in c("ANANEA", "ANDAHUAYLAS")) {
  p_est <- graficar_nspei_multiescala(df_nspei, est)

  nombre_base <- paste0(
    "nspei_multiescala_",
    gsub("[^A-Za-z0-9]+", "_", est)
  )

  ggsave(
    filename = file.path(ruta_figuras, paste0(nombre_base, ".png")),
    plot = p_est,
    width = 16,
    height = 10,
    dpi = 300,
    bg = "white"
  )
}


## 7. Diagrama de Hovmöller del nSPEI-12


In [ ]:
#========================================================
# 2. ORDEN DE ESTACIONES
#   Aquí puedes cambiar a orden por altitud si quieres
#========================================================
orden_estaciones <- df_nspei %>%
  distinct(estacion) %>%
  arrange(estacion) %>%
  pull(estacion)

#========================================================
# 3. FUNCIÓN PARA CLASIFICAR CATEGORÍAS
#========================================================
clasificar_nspei <- function(x) {
  case_when(
    is.na(x)        ~ NA_character_,
    x <= -2.0       ~ "D4",
    x <= -1.6       ~ "D3",
    x <= -1.3       ~ "D2",
    x <= -0.8       ~ "D1",
    x <  -0.5       ~ "D0",
    x <=  0.5       ~ "Normal",
    x <   0.8       ~ "W0",
    x <   1.3       ~ "W1",
    x <   1.6       ~ "W2",
    x <   2.0       ~ "W3",
    x >=  2.0       ~ "W4"
  )
}

#========================================================
# 4. PALETA OFICIAL
#========================================================


colores_nspei <- c(
  "D4"     = "#5B0F14",
  "D3"     = "#8B1A1A",
  "D2"     = "#FC8D59",
  "D1"     = "#E6A57E",
  "D0"     = "#F4E4A1",
  "Normal" = "#F7F7F7",
  "W0"     = "#CDECF2",
  "W1"     = "#8ECDE6",
  "W2"     = "#4A90C2",
  "W3"     = "#2C5AA0",
  "W4"     = "#1B3A6F"
)



#========================================================
# 5. FUNCIÓN DE GRAFICADO POR ESCALA
#========================================================
graficar_hovmoller_escala <- function(df_base, var_escala, etiqueta_escala) {

  df_plot <- df_base %>%
    select(fecha, estacion, all_of(var_escala)) %>%
    rename(nspei = all_of(var_escala)) %>%
    filter(fecha >= as.Date("1970-01-01"),
           fecha <= as.Date("2023-12-31")) %>%
    mutate(
      estacion = factor(estacion, levels = orden_estaciones),
      categoria = clasificar_nspei(nspei),
      categoria = factor(
        categoria,
        levels = c("D4", "D3", "D2", "D1", "D0",
                   "Normal",
                   "W0", "W1", "W2", "W3", "W4")
      )
    )

  p <- ggplot(df_plot, aes(x = estacion, y = fecha, fill = categoria)) +
    geom_tile(color = NA) +
    scale_fill_manual(
      values = colores_nspei,
      drop = FALSE,
      na.value = "grey85",
      name = NULL
    ) +
    scale_y_date(
      limits = as.Date(c("2023-12-31", "1970-01-01")),
      date_breaks = "1 year",
      date_labels = "%Y",
      expand = c(0, 0)
    ) +
    labs(
      title = paste0("Evolución espacio-temporal del ", etiqueta_escala, " (1970-2023)"),
      subtitle = paste0("Región Sur del Perú - Escala de ", gsub("nSPEI-", "", etiqueta_escala), " meses"),
      x = "Estaciones meteorológicas",
      y = "Cronología anual"
    ) +
    guides(
      fill = guide_legend(
        nrow = 1,
        byrow = TRUE,
        override.aes = list(color = NA)
      )
    ) +
    theme_minimal(base_family = "serif", base_size = 12) +
    theme(
      plot.title = element_text(face = "bold", size = 18, hjust = 0.5),
      plot.subtitle = element_text(size = 12, hjust = 0.5),
      axis.title.x = element_text(face = "bold", size = 13),
      axis.title.y = element_text(face = "bold", size = 13),
      axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, size = 10),
      axis.text.y = element_text(size = 7),
      panel.grid = element_blank(),
      legend.position = "bottom",
      legend.direction = "horizontal",
      legend.box = "horizontal",
      legend.text = element_text(size = 10),
      legend.key.width = unit(1.8, "lines"),
      legend.key.height = unit(0.9, "lines"),
      plot.margin = margin(15, 15, 15, 15)
    )

  return(p)
}


In [ ]:
p_12 <- graficar_hovmoller_escala(
  df_nspei, "nspei_12", "nSPEI-12"
)  # Genera el Hovmöller de la escala anual.

ggsave(
  filename = file.path(ruta_figuras, "Hovmoller_nSPEI_12.png"),
  plot = p_12,
  width = 11,
  height = 14,
  dpi = 400,
  bg = "white"
)


In [ ]:
# ============================================================
# VISUALIZACIÓN MULTIESCALA DEL nSPEI PARA LAS 28 ESTACIONES
# ============================================================

estaciones <- sort(unique(df_nspei$estacion))  # Obtiene y ordena las 28 estaciones.

for (est in estaciones) {                      # Recorre cada estación meteorológica.

  p_est <- graficar_nspei_multiescala(
    df_nspei,
    est
  )                                            # Genera nSPEI-3, nSPEI-6 y nSPEI-12.

  print(p_est)                                 # Muestra la figura dentro del notebook.
}

## 8. Identificación de eventos de sequía


In [ ]:
extraer_eventos <- function(df_est, var_nspei, umbral_inicio = 0, umbral_sequia = -1) {

  x <- df_est[[var_nspei]]
  fechas <- df_est$fecha

  en_evento <- !is.na(x) & x < umbral_inicio               # Identifica rachas con nSPEI < 0.

  r <- rle(en_evento)                                       # Detecta secuencias consecutivas.
  fin <- cumsum(r$lengths)
  ini <- fin - r$lengths + 1

  eventos <- tibble()

  for (i in seq_along(r$values)) {
    if (r$values[i]) {
      idx <- ini[i]:fin[i]
      vals <- x[idx]

      if (min(vals, na.rm = TRUE) <= umbral_sequia) {       # Retiene rachas que alcanzan nSPEI <= -1.
        eventos <- bind_rows(eventos, tibble(
          estacion = unique(df_est$estacion),
          cluster_ward = unique(df_est$cluster_ward),
          escala = var_nspei,
          fecha_inicio = min(fechas[idx], na.rm = TRUE),
          fecha_fin = max(fechas[idx], na.rm = TRUE),
          duracion = length(idx),                            # Duración del evento en meses.
          severidad = sum(-vals, na.rm = TRUE),              # Déficit acumulado del evento.
          intensidad = sum(-vals, na.rm = TRUE) / length(idx), # Severidad media mensual.
          minimo_nspei = min(vals, na.rm = TRUE)             # Mínimo nSPEI alcanzado.
        ))
      }
    }
  }

  eventos
}

eventos_3 <- df_nspei %>%
  group_by(estacion) %>%
  group_split() %>%
  map_dfr(~ extraer_eventos(.x, "nspei_3", umbral_inicio = 0, umbral_sequia = -1))

eventos_6 <- df_nspei %>%
  group_by(estacion) %>%
  group_split() %>%
  map_dfr(~ extraer_eventos(.x, "nspei_6", umbral_inicio = 0, umbral_sequia = -1))

eventos_12 <- df_nspei %>%
  group_by(estacion) %>%
  group_split() %>%
  map_dfr(~ extraer_eventos(.x, "nspei_12", umbral_inicio = 0, umbral_sequia = -1))

eventos_all <- bind_rows(eventos_3, eventos_6, eventos_12)   # Integra las tres escalas.

write.csv(
  eventos_all,
  file.path(ruta_procesados, "eventos_nspei_todas_escalas.csv"),
  row.names = FALSE
)


## 9. Tablas de eventos de sequía


In [ ]:
conteo_estacion <- eventos_all %>%
  group_by(estacion, cluster_ward, escala) %>%
  summarise(
    n_eventos = n(),
    duracion_media = mean(duracion, na.rm = TRUE),
    duracion_mediana = median(duracion, na.rm = TRUE),
    severidad_media = mean(severidad, na.rm = TRUE),
    severidad_mediana = median(severidad, na.rm = TRUE),
    intensidad_media = mean(intensidad, na.rm = TRUE),
    minimo_medio = mean(minimo_nspei, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(escala, desc(n_eventos))

conteo_cluster <- eventos_all %>%
  group_by(cluster_ward, escala) %>%
  summarise(
    n_eventos = n(),
    duracion_media = mean(duracion, na.rm = TRUE),
    duracion_mediana = median(duracion, na.rm = TRUE),
    severidad_media = mean(severidad, na.rm = TRUE),
    severidad_mediana = median(severidad, na.rm = TRUE),
    intensidad_media = mean(intensidad, na.rm = TRUE),
    minimo_medio = mean(minimo_nspei, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  arrange(escala, cluster_ward)

estaciones_pocos_eventos <- conteo_estacion %>%
  mutate(
    criterio_copula = case_when(
      n_eventos < 10 ~ "Muy insuficiente",
      n_eventos < 15 ~ "Insuficiente",
      n_eventos < 20 ~ "Limitado",
      TRUE ~ "Aceptable"
    )
  ) %>%
  arrange(escala, n_eventos)

write.csv(
  conteo_estacion,
  file.path(ruta_tablas, "conteo_eventos_por_estacion_y_escala.csv"),
  row.names = FALSE
)

write.csv(
  conteo_cluster,
  file.path(ruta_tablas, "conteo_eventos_por_cluster_y_escala.csv"),
  row.names = FALSE
)

write.csv(
  estaciones_pocos_eventos,
  file.path(ruta_tablas, "diagnostico_suficiencia_eventos_por_estacion.csv"),
  row.names = FALSE
)


## 10. Clasificación de eventos


In [ ]:
eventos_all <- eventos_all %>%
  mutate(
    clase_evento = case_when(
      minimo_nspei <= -2.0 ~ "D4 - Excepcional",
      minimo_nspei <= -1.6 ~ "D3 - Extrema",
      minimo_nspei <= -1.3 ~ "D2 - Severa",
      minimo_nspei <= -1.0 ~ "D1 - Moderada",
      TRUE                  ~ "No clasificada"
    ),
    clase_evento = factor(
      clase_evento,
      levels = c(
        "D1 - Moderada",
        "D2 - Severa",
        "D3 - Extrema",
        "D4 - Excepcional",
        "No clasificada"
      )
    )
  )

frecuencia_clases <- eventos_all %>%
  group_by(escala, clase_evento) %>%
  summarise(
    n_eventos = n(),
    .groups = "drop"
  ) %>%
  arrange(escala, clase_evento)

print(frecuencia_clases)

write.csv(
  eventos_all,
  file.path(
    ruta_procesados,
    "eventos_nspei_todas_escalas_clasificados.csv"
  ),
  row.names = FALSE
)


## 11. Figuras de eventos de sequía utilizadas en la tesis


In [ ]:
p_cluster <- ggplot(conteo_cluster, aes(x = cluster_ward, y = n_eventos, fill = escala)) +
  geom_col(position = "dodge") +
  theme_minimal() +
  labs(
    title = "Número de eventos de sequía por cluster y escala temporal",
    x = "Cluster",
    y = "Número de eventos"
  )

ggsave(
  file.path(ruta_figuras, "numero_eventos_por_cluster_y_escala.png"),
  plot = p_cluster,
  width = 9,
  height = 6,
  dpi = 300
)

p_clases <- ggplot(frecuencia_clases, aes(x = escala, y = n_eventos, fill = clase_evento)) +
  geom_col(position = "stack") +
  theme_minimal() +
  labs(
    title = "Distribución de clases de eventos de sequía por escala temporal",
    x = "Escala",
    y = "Número de eventos",
    fill = "Clase"
  )

ggsave(
  file.path(ruta_figuras, "clases_eventos_por_escala.png"),
  plot = p_clases,
  width = 9,
  height = 6,
  dpi = 300
)

p_estacion <- ggplot(conteo_estacion, aes(x = estacion, y = n_eventos, fill = escala)) +
  geom_col(position = "dodge") +
  coord_flip() +
  theme_minimal() +
  labs(
    title = "Número de eventos de sequía por estación y escala temporal",
    x = "Estación",
    y = "Número de eventos"
  )

ggsave(
  file.path(ruta_figuras, "numero_eventos_por_estacion_y_escala.png"),
  plot = p_estacion,
  width = 11,
  height = 8,
  dpi = 300
)


In [ ]:
p_cluster <- ggplot(
  conteo_cluster,
  aes(x = cluster_ward, y = n_eventos, fill = escala)
) +
  geom_col(position = "dodge") +
  theme_minimal() +
  labs(
    title = "Número de eventos de sequía por cluster y escala temporal",
    x = "Cluster",
    y = "Número de eventos"
  )

ggsave(
  file.path(ruta_figuras, "numero_eventos_por_cluster_y_escala.png"),
  plot = p_cluster,
  width = 9,
  height = 6,
  dpi = 300
)

p_cluster

In [ ]:
p_clases <- ggplot(
  frecuencia_clases,
  aes(x = escala, y = n_eventos, fill = clase_evento)
) +
  geom_col(position = "stack") +
  theme_minimal() +
  labs(
    title = "Distribución de clases de eventos de sequía por escala temporal",
    x = "Escala",
    y = "Número de eventos",
    fill = "Clase"
  )

ggsave(
  file.path(ruta_figuras, "clases_eventos_por_escala.png"),
  plot = p_clases,
  width = 9,
  height = 6,
  dpi = 300
)

p_clases

In [ ]:
p_estacion <- ggplot(
  conteo_estacion,
  aes(x = estacion, y = n_eventos, fill = escala)
) +
  geom_col(position = "dodge") +
  coord_flip() +
  theme_minimal() +
  labs(
    title = "Número de eventos de sequía por estación y escala temporal",
    x = "Estación",
    y = "Número de eventos"
  )

ggsave(
  file.path(ruta_figuras, "numero_eventos_por_estacion_y_escala.png"),
  plot = p_estacion,
  width = 11,
  height = 8,
  dpi = 300
)

p_estacion